In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("concrete.csv")
df

,cement,slag,ash,water,superplastic,coarseagg,fineagg,age,strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [4]:
# separer X et yy
X = df.drop("strength", axis=1)
y = df["strength"]

In [6]:
y

,strength
0,79.99
1,61.89
2,40.27
3,41.05
4,44.30
...,...
1025,44.28
1026,31.18
1027,23.70
1028,32.77


In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# standardiser
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
# transformation des array en tenseurs avec pytorch
import torch
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)

/tmp/ipython-input-1534372913.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
/tmp/ipython-input-1534372913.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)


In [13]:
# MODEL ARCHITECTURE
class linearRegression(torch.nn.Module):
    def __init__(self, input_dim):
        super(linearRegression, self).__init__()
        self.linear = torch.nn.Linear(input_dim, 10) # la première couche prends les input et passe les nombre de neuronnes à 10
        self.linear2 = torch.nn.Linear(10, 5) # la seconde couche prends les input
        self.linear3 = torch.nn.Linear(5, 3) # la dernière couche prends les input
        self.linear4 = torch.nn.Linear(3, 1) # la dernière couche prends les input 1

    # feed forward, on ajoute les fonctions d'activations
    def forward(self, dimension):
        out = torch.relu(self.linear(dimension))
        out = torch.relu(self.linear2(out))
        out = torch.relu(self.linear3(out))
        out = self.linear4(out)
        return out

input_dim = X_train.shape[1]
# initialisation des poids au hasard
torch.manual_seed(42)
model = linearRegression(input_dim)
# ajouter l'optimiseur
loss = torch.nn.MSELoss()
# optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
# optimiser Adams
optimizer = torch.optim.Adam(params = model.parameters(), lr=0.01)

# training the model
num_of_epochs = 10000
for i in range(num_of_epochs):
  # Give the input data to the architecture
  # Initialisation
  y_train_prediction = model(X_train)
  # enlever la dimension du batch size dans la prediction
  loss_value = loss(y_train_prediction.squeeze(), y_train)
  # nettoyer la mémoire, retirer les gradiants après chaque itération
  optimizer.zero_grad()
  # Backpropagation
  loss_value.backward()
  # update des weights
  optimizer.step()

  # print du loss
  if i % 100 == 0:
    print(f"epoch: {i} loss value for training part =: {loss_value}")


epoch: 0 loss value for training part =: 1604.9345703125
epoch: 100 loss value for training part =: 201.76705932617188
epoch: 200 loss value for training part =: 126.511474609375
epoch: 300 loss value for training part =: 110.7541732788086
epoch: 400 loss value for training part =: 106.6697006225586
epoch: 500 loss value for training part =: 105.38582611083984
epoch: 600 loss value for training part =: 104.59159088134766
epoch: 700 loss value for training part =: 104.001953125
epoch: 800 loss value for training part =: 103.50282287597656
epoch: 900 loss value for training part =: 103.00176239013672
epoch: 1000 loss value for training part =: 102.40140533447266
epoch: 1100 loss value for training part =: 101.68070983886719
epoch: 1200 loss value for training part =: 100.95413970947266
epoch: 1300 loss value for training part =: 100.45563507080078
epoch: 1400 loss value for training part =: 100.01271057128906
epoch: 1500 loss value for training part =: 97.25776672363281
epoch: 1600 loss 

In [14]:
# évaluation et performance
# torch no grad (le modèle est déja pret, je veux juste teste, pas de gradiant)
with torch.no_grad():
  # model en mode evaluation
  model.eval()
  y_test_prediction = model(X_test)
  test_loss = loss(y_test_prediction.squeeze(), y_test)
  print(f"test loss value: {test_loss.item(): 4f}")

test loss value:  71.997612


In [ ]:
# sauvegarder le modèle
from pathlib import Path
filename = Path('models')
model_name = 'linear_regression.pth'
filename.mkdir(parents=True, exist_ok=True)

#Save Path avec uniquement ls poids du modèle
saving_path = filename / model_name
print(f"Saving model to: {saving_path}")
torch.save(obj=model.state_dict(), f=saving_path)

Saving model to: models/linear_regression.pth


In [17]:
# charger le modèle
loaded_model = linearRegression(input_dim)
loaded_model.load_state_dict(torch.load(f=saving_path))

<All keys matched successfully>

In [21]:
# prediction sur le modele
loaded_model.eval()
with torch.no_grad():
  pred = loaded_model(torch.tensor([[1.45,2.33,3.78,45,5.87,6.0,7.0,8.0]]))
  print(f"Prediction value: {pred.item()}")

Prediction value: 1643.534912109375
